# Session 1: Single Spatial View

# What is **`vitessce`** ?

## Motivation

So far in the tutorial we have provided a general overview of Vitessce and introduced its key features:

- _Modular grid of views_
- _Supported data formats_
- _Coordinated interactivity_

Although Vitessce is very flexible, the internal JSON-based configuration is not ergonomic to write by hand and deploying a Vitessce-based visualization requires serving data files via a web server. Together, these challenges serve as a barrier to entry for some, and we were motivated to design a simplified API for computational biologists to visualize their own datasets with Vitessce. 

Enter **`vitessce`**...

## Overview

**`vitessce`** is a **Python package** designed to create interactive visualizations of single-cell and spatial omics data. Its main features include:

- Authoring Vitessce visualizations by configuring the data, views, and view linkages.

- Displaying Vitessce visualizations directly in computational notebooks (Jupyter, JupyterLab, Google Colab, Marimo)

- Transparently hosting local data files for visualizations (hiding web server complexities)


## How it works

The **`vitessce`** Python library exposes a simple API that maps directly to the internal Vitessce JSON configuration format. Users write Python programs with **`vitessce`** which ultimately:

- Emit JSON (the Vitessce visualization configuration)
- Render the visualization within computational notebooks as a widget 

# Getting started

The remainder of this notebook will focus on introducing the Vitessce Python package and its API.

Start by importing the `VitessceConfig` class from `vitessce`.

In [ ]:
!pip install "vitessce[all]==3.9.2"

In [ ]:
from vitessce import VitessceConfig

### 1. Introduction to Vitessce
Vitessce is an open-source framework for interactive visualization of spatial single-cell data. It uses a **VitessceConfig** class to define which data to load and how to display it in different 'views' (like a heatmap or a scatterplot).

### 2. Defining a configuration
We start by initializing a `VitessceConfig` object.
Then we call `.add_dataset()` and `.add_file()` (chained together) to define a new dataset and add a file to that dataset (that we just defined).

In [ ]:
from vitessce import VitessceConfig, ViewType as vt, DataType as dt, FileType as ft

# 1. Initialize the config with a name
config = VitessceConfig(schema_version="1.0.18", name="Single Spatial View Tutorial")

# 2. Add a dataset
# We'll use a public example dataset URL for this tutorial.
# Note that we use this `.add_file` method to specify a URL to a remotely-hosted data file.
dataset = config.add_dataset(name="Kuppe et al 2022").add_file(
    url="https://vitessce-data.storage.googleapis.com/0.0.33/main/kuppe-2022/kuppe_2022_nature.visium.ome.zarr",
    file_type="image.ome-zarr"
)

# 3. Add a Spatial View to the configuration
spatial_view = config.add_view("spatialBeta", dataset=dataset)

# 4. Arrange the layout (single view filling the whole widget)
config.layout(spatial_view)

print("Configuration created successfully!")

### 3. Rendering the Widget
Finally, we call `.widget()` to render the interactive dashboard directly in Colab.

In [ ]:
my_widget = config.widget()
my_widget

Notes

*   We used `.add_file` to specify a URL to a file that was already remotely hosted on the internet publicly. How might we specify a dataset that is on our local machine or on a connected compute cluster? ([docs](https://python-docs.vitessce.io/data_options.html))
*   Where can I find a list of the valid `file_type` values? ([docs](https://vitessce.io/docs/data-types-file-types/))

*   Where can I find a list of the valid `view_type` values? ([docs](https://vitessce.io/docs/components/))

* If you modify the configuration code, be sure to run BOTH the `config = VitessceConfig` and the `config.widget()` notebook cells (and any additional cells in between). If you are not sure why, please let us know and we can explain.



#### Looking under the hood: the initial JSON configuration

In [ ]:
# We can view the internal JSON representation of the configuration with the `.to_dict` method.
# (The `base_url` parameter allows for specifying a URL prefix, which is only relevant when using local data objects that do not yet have remote URLs)
config.to_dict(base_url="")

#### Looking under the hood: the current JSON configuration

In [ ]:
my_widget._config

In [ ]:
# Try it: Run the cell, then zoom in or out of the image in the widget above. Then re-run this code cell.
my_widget._config["coordinationSpace"]["spatialZoom"]["A"]

## Specifying initial visualization properties

By default, Vitessce fits the data (for spatial/imaging and scatterplot views) in the viewport on first load.
You can override such initial settings by specifying **initial coordination values**.

The concept of "coordination" comes from Vitessce's support for "coordinated multiple views", which means that visualization properties can be linked across views. The support for coordinated properties __also serves as the mechanism to specify initial visualization properties.__

Therefore, despite the current notebook focusing on the single-view case, we briefly introduce the concept of coordination as a prerequisite for specifying initial settings (or overriding their defaults).

Almost all of the state (visualization properties, interaction settings, etc.) in Vitessce are represented as variables in the **"coordination space"** part of the Vitessce configuration. Each of these variables has a type ("coordination type"), a name ("coordination scope"), and a value ("coordination value"). Think of it like defining variables in a statically-typed programming language:

```c
int x = 99;         // C variables
spatialZoom y = 63; // Vitessce coordination values (In reality, these are specified via a JSON representation)
```

Examples of coordination types used by the spatial view are:

| Coordination Type | Description | Example Value |
|---|---|---|
| `spatialZoom` | Zoom level (Higher = more zoomed-in; Log scale) | `2` |
| `spatialTargetX` | X-coordinate of the viewport center | `500.0` |
| `spatialTargetY` | Y-coordinate of the viewport center | `500.0` |
| `featureSelection` | List of selected genes or other features | `["EPCAM"]` |

Use `config.link_views()` to pin initial values for one or more coordination types across a list of views:

```python
config.link_views(
    views=[my_view],
    c_types=["spatialZoom", "spatialTargetX", "spatialTargetY"],
    c_values=[2, 500.0, 500.0]
)
```

In [ ]:
from vitessce import VitessceConfig

config2 = VitessceConfig(schema_version="1.0.18", name="Initial Properties Example")

dataset2 = config2.add_dataset(name="Kuppe et al 2022").add_file(
    url="https://vitessce-data.storage.googleapis.com/0.0.33/main/kuppe-2022/kuppe_2022_nature.visium.ome.zarr",
    file_type="image.ome-zarr"
)

spatial_view2 = config2.add_view("spatialBeta", dataset=dataset2)

# Set the initial zoom level and pan position
config2.link_views(
    views=[spatial_view2],
    c_types=["spatialZoom", "spatialTargetX", "spatialTargetY"],
    c_values=[-2, 300.0, 300.0]
)

config2.layout(spatial_view2)
config2.widget()

### Exercise 1

👉 **Modify the `config2` cell above** to:

- **Change** the `spatialZoom` value to `1` (zoomed in)
- **Change** `spatialTargetX` and `spatialTargetY` to pan to the center of the tissue
- **Disable** the R, G, B channel labels (hint: see `spatialChannelLabelsVisible`)

After making your changes, re-run the cell to see the updated widget.

Question: What does the "Click to recenter" button do now?

## Using local data with SpatialData

### What is SpatialData?

[SpatialData](https://spatialdata.scverse.org/) is a Python framework for storing and manipulating spatial omics data. A `SpatialData` object bundles together:

- **Images**: tissue photographs or microscopy images (stored as OME-Zarr)
- **Labels**: segmentation masks identifying individual cells (bitmask images)
- **Shapes**: geometric observations such as Visium spots (circles) or cell outlines (polygons)
- **Points**: individual molecule coordinates, e.g. from Xenium/MERFISH experiments
- **Tables**: AnnData objects linking observations (spots, cells) to gene expression data

Vitessce can read SpatialData objects via the `SpatialDataWrapper` class.

### Download the example Visium dataset

The cell below downloads a small Visium breast-cancer dataset as a `.zarr` zip archive and unzips it locally. The download only runs once — subsequent runs detect the existing directory and skip the download.

In [ ]:
!pip install "spatialdata==0.7.3"

In [ ]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import zipfile

data_dir = "data"
zip_filepath = join(data_dir, "visium.spatialdata.zarr.zip")
sdata_filepath = join(data_dir, "visium.spatialdata.zarr")

if not isdir(sdata_filepath):
    if not isfile(zip_filepath):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/visium.spatialdata.zarr.zip",
            zip_filepath
        )
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_filepath)

print("Dataset ready at:", sdata_filepath)

### Inspect the SpatialData object

Let's load the dataset and look at what it contains.

In [ ]:
from spatialdata import read_zarr

sdata = read_zarr(sdata_filepath)
sdata

SpatialData objects may not contain all 5 types of spatial elements. For example, this object only contains Images, Shapes, and Tables.

Question: how many coordinate systems does it contain?

Question: How do these elements relate to the Zarr folder contents?

In [ ]:
# os.listdir(join(sdata_filepath, "images"))

In [ ]:
# os.listdir(join(sdata_filepath, "shapes"))

In [ ]:
# os.listdir(join(sdata_filepath, "tables"))

In [ ]:
# os.listdir(join(sdata_filepath, "tables", "table"))

### Configure Vitessce for image + spots

The `SpatialDataWrapper` maps paths within the SpatialData Zarr store to Vitessce data types. We pass:

- `image_path`: the relative path to a particular image within the Zarr store
- `obs_spots_path`: the relative path to the shapes element that corresponds to the Visium spots (as circles)
- `obs_feature_matrix_path`: the relative path to the expression matrix array within the Zarr store (will always be within a Table element)
- `coordinate_system`: the coordinate system to use for alignment of the spatial layers

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    SpatialDataWrapper,
)

vc = VitessceConfig(schema_version="1.0.18", name="Visium Spatial Data")

wrapper = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    image_path="images/ST8059050_hires_image",
    obs_spots_path="shapes/ST8059050",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    coordinate_system="ST8059050",
    coordination_values={
        "obsType": "spot" # Since this is a Visium dataset, we specify an alternative "obsType" (the default is "cell")
    },
)
dataset = vc.add_dataset(name="Visium Breast Cancer").add_object(wrapper)

# Add a spatial view, a layer controller, and a gene list
spatial = vc.add_view("spatialBeta", dataset=dataset)

# Link all views to share the same obsType coordination value
vc.link_views([spatial], ["obsType"], ["spot"])

vc.layout(spatial)
vc.widget()

#### Exercise

- Try updating the SpatialDataWrapper parameters to view the other Visium sample in the dataset.

## Multiplexed imaging

So far we have used an RGB (e.g., histology) image. Vitessce also supports visualization of **multiplexed (multi-channel) images** in which each channel represents a different protein marker or staining. The individual channels can be thought of as greyscale images, but we use **pseudocoloring** (mapping each channel to a different color) to visualize multiple channels at a time.

When Vitessce detects that an image is a multiplex (non-RGB) image, it will assign a set of default colors to each channel. 
Vitessce will also initialize each channel's intensity window based on the distribution of intensity values (i.e., using the interquartile range).


Because Vitessce can display multiple images at a time (i.e., multiple image _layers_), the visualization properties for imaging data are represented using a hierarchical data structure.

In [ ]:
from IPython.core.display import HTML
display(HTML("""
<?xml version="1.0" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN" "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<svg version="1.1" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 525.5 568.5" width="300px"><!-- svg-source:excalidraw --><metadata></metadata><defs><style class="style-fonts">
      @font-face { font-family: Excalifont; src: url(data:font/woff2;base64,d09GMgABAAAAABqcAA4AAAAALpgAABpEAAEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGigbiTocghQGYACBPBEICsZMs2kLXAABNgIkA4E0BCAFgxgHIBvoI6OQtBLtJ/vLgzjG4DqkutLMqwmv3FmRtmmaZjauSCR9zkfiKaW+tg45lf/OlaDox357994X1CxBMinZIpVSxEMhNKZrqJTiBMClXrJ1USg6NgVX59hY8oqVtCXIH+qs7C+Tlo6hesIKKbv5ovumldpRO6OxrayzINuh5cDSMcv211ffEH/q33svcQO3jps0rZOCEYqg4LqUSsGRxSB2UscOwheG/9e5/Q/AHn543eyTNW+1CIVKtHIKJ6YETia+jH/Yr2u7xgh2EsTa/isUTsSTvC8Wja2jWjPXprszSKJVYqDEN/n7tVT/ADkgiTsRJkJFRZgIhcL+e4cv70hlZn8I90K4cXcbFI5AqEzGMyvEVvVcgVyFaV1lq0y9bY3x9d/hss7NX0MWslA99itrqBgwsdYIZEOBxVidrZCUaH/GLp9E2ThTv3nRgaGJBbn917uLIOb47vA85mL34aqdCHAkLc4P764ke1SDOBKLS5bmXDi8QZz7oeXn8pVA0v87/5c9/8/LVPb/3NuAJ7SXy3kXaJPEhrs+MBicM91L7PMA4CD/lAGLZdzCkP+7ZmCt71vsWyz4f7vdIEDhRjKLXgAEvQgwg7sKErLlypOvQKESpcpVqFSjVoNGTdq0G4pinVLh5JQ0QM5cYsKcUC6WSNVBkahYPsMMf/mEpEwFcmUpNE5EiaKIDnkttfRjpmCAcZCQ5UgOHp4hU8qSJbsgR65GZC4/y2S01lq5tSKSNyy0tqlo1njSJjwyaB1nTVJSiokFMncDxYAzypiCAziSmrIAdyyGWhETZbciua1QSRVkBk/SOmzfHpZQfXF6vCt/fXeggA+aXNnl1j8QvS+XcO9LS6fsepJeXGIoYQeHjWrIVaJBUSOHUlOIs2aZQga1NfbsG/NyNdSmVA4HkDykLXMhmAycclVUudwO5GAIwxYktemsxHAr1CuSp8MQXoHV2WhL0xeklg2X54en/Kp3Gmntg9j9sqHWPC4AfffjqHf4GDgVDw4iNHqUW9UzTN7fRa9BIyYtGTVtzrJV6zaduvHo2Wugp47d+g2boPgvJe66cPe7xBRyziuP3HXDNXt27ZgZ27Jpw3ohKFDgMPr2fNk+TCPrh5wvIzYnkEaamM0lTzWBNgtPFBGEey8v1C7pfE5I9pi6UCaTaS72kdbHN6pNuRFMMUcRT5YyHZkaT29RssbDN4CiUOZZ0kzhfgS2H8vAJzDaiDlgZbVEuFaqES65mP0SGQKFyPZQECZcEPFmQFzKj9UBMWcZqOP0DtGwjvp8sAsbG7iCg4Fp1TTVJcO9D3xUtFG4q1cY5/TKfdxYTsaUBkqyjWcHngBPbOryq29xX3jyy+34wNpHDtK4BR3RSe7b/FEADA4bsTIl/0/5NnyftE+ldcJU41Kg828SKIWKjNdqHWqFujXtRocM6En1CLX5WXc9tfJyDwSDoMUHEPGXvWQ5lgPtQ3Ht4I1PZx/YlEF2u9iv3EkEa99b+iiEuWsSKckHnnQYcNDYwMWPnDsoWNxoRCnflzNHG5Uir33HX6yxXTrXHmb6vJu6LPI3U2TUybaYqnKXgTqpmqDpkpLmkZL1eeIRbvAB358QPXgOYCXwmECoExL63IUpnu7RnF7B0bqiEKHdYSZ51527uh8fGDRUm7bpMQfqYGY9f1a6LjFYSpL4PMdEKfXcpnZenBlJhReyhd/a+K3bmwcPYvYY8h6i2c4HHTV80l9YjRJO3N0+lGIqtl92cWyNgyDA91/KA7r3zi9nkXGZimVGCJ7wmM5Wv64bTImplVK60RhwThDezOWqO3grZR4p1WSKWhDLjGjd0B2yvJeBmsAyRO+IvNUFiA/sxcmCJfczH1dVnVS5RDKBBEK3zLGAm+KGxo14OsK604AwCFpWDzrNlZCiWgbUG0S3YA86HW86286IoVIZ1xwMOOQAp0n9TMuUaka4SzJU5abr2qia9HkMgvvlljiQAZ5DDqyJeOttTxA2KluqbtdtRB60WYAfITfWKxeC179DnCtY/u6qGVUoUh59q+gzx0TWfYmDGjVHZPjo26tAGV+L90w52IctxgZj/mCiN+QhmgqftI3+X4D3jgzi2eeIpKhEOHMto96NRtrwvvoYLd6KLFXsIuOUIrSQCB7jsbicKxHludskzS3JfFL66F5KoY8J5EpRnyHPa1ljaAch9PooiZR6va6Yy31BVDyIBbsh6mQUJ+NY1P4KjqV1hSCKKO/+veB1UrIsZCQbopBHBh/DVifkRZlBiKadbWi2KcU5mYxZnfTNM4UX3qOpBSFEdSJN3lwGjGSIAQrVifE6jBcCltY3b6dW8h9+2FCSdMlR87spQaDO/AY/yjonhr3nfKCjqnE62xhMH5EObPV0tAITsp0kFbvVZEOD83M3HN+p4iIsYyy7YIb6f0dCaCRNjSkpyWSLPtZE6+lwIUxgyAcFVFMO5DG2J+67NfZZGub5Xk0yqRWxSCxvi2a4iz5tf3ov15GtmmbBggi90Mxj+bnltaOdrCVf+LAlz+lZ7ROzs5pnZkTGbNtQGtY+vi8gW9bORhLjVsAkn5EZShTig0kXFVlniAmh50yQQA7gWNWy4Gy5GnLVijVtPKutg4VEWz7by+QGUCbVbVhPEG2HehXV0YJ2hybcHm5EOfcFpT2Z3InIGrahbD/35nsL/cJdFCgibUNn2HWJSR25E/JOhzsf2NTpSZQI11q6Q5kwL+ejyzm0ENq3bLw16J4DBksbuJiv5buvDt6gmexB2wiLfAAhcHCM9QYcf58X5SIKTyeR1qY2I3aNTZjVOIolv/zVSSQ0T7ZP8r1SjRjKGo0eoHCqJbJByHuH3w/7EQph4JXq9dzNpNveRecJUKQkNrhi6bhMj5ZsZujed2wpPGY5OHMhYJaik5OzF7OTmeAdpM+5tkPBBJBdX14G2TPJnLkA45Z+aG5JdOzohUXjIhJ1FR6S+TjfvTrcyyMyN0E7+yii6MFA1R2cpiHA0zDrYla79FGOWitx7BhPOZC9Q14nEoQV/PkYaaMjxto7WZcXbUWRbJ1aknPetcKJvTi5QvfuA3VYQMEB1ifScU9HiNZCekECFP0WKDvVZKHeWAh5J521+ZGMBVDXYrFI28+tEI52IQebNCv9RZd06d1YXIlnzmrXdW0b9HCIXEQ0WTIDyrgWTeVBYWGHm9j4iNOqZB4p5No4JaL9VIb/2tkB1nHsjiSTIUFUineQKHtMC6cJtTe2UvXvsis0tUynDVCGsNEcMYthaMKoUhEIojkWtc/6N6doo8PB1J3IUNxXkgVvSKV1bWJsmkGlYg/kAKev6lR17rIRhy0cDda+qmHLgraBNetXNHSChTerxkVcV4Xc4aAIZvhPpn7jG0qkqFnZRUoIGVFmeHWVcLdLhju53reF3MyJChNCDMz7z4crybZeJPv5faCiw3nVCQ6Ct3lFaUT7vtPBhRWghdiCOzx6FYb6KLmOluM4uICMF2mNSiu/zmZAKbm+kk54YhwfqIiQcRryvZtqc4nB22M72XD3bia5ub33xskHvT4jYI+DBq6M9UsjiD24KFDR8iyIHDwV5+NcLBmN64hsRphtLpF7IxoO6++A0riE0xgX0B0fx3v/KKnqy+tzews6vLYGQQDRQ7uRkRuRvfvTnxams/p+VSoSJhlhC2GAz/exdY18OTvZ9IeZvkAROp+9a2Y+niR69lxAumaCHcQC1TUGEvphjMK2cHgRjlqfT+GgAI0GdpSwSk4n8kgd5ymO7vqUZdbXM9wlPAQj3ao2YhYXqG4DNVfyZNdl3A1gu476CQE7t6KJBR1NCJYcPNgW3b0rHpE+K31n0d2sW2wjzwsnCHB3lTTNOjEnlV0sFtGqQMLF4Ssa4O18kN066X+KuHXBMfcc7v+B11h/PW9o6uaXFWb3ctHiPSp1gUJ9HvVFM3yicLNbWjU1F5dclxz/W+wfaqHHYAg74Ra0LI8P5P9SX8QxfGiFn6ycGza1bARmQDq2pmkcBA2HNBOjprb2M7VEBQat7GfQ4as0in86N4XdBkr75ay73f2pF5TfgOqDRyOqoNVsbRQ2deUVFeVxdD0VDxHK64DyzsQBLRRnk2Mty1in9omxJFu1YpW+s4sTdRU13420MtdfS38PSzCVLfdpmEffegdO6rfecbjkfj7CX4v/TxC0qv/2kY9DJPEe+bGTk9Fh3igiwz/aT0I650pgIRJmA5XhWgflR1VMUQIqlR+0Zdt8OMYPHObIHuzrsWvWFZ3NeVU2oZ891r/3ohbMFk1YVctnW6CO4l/EA2nEwsKzAFP9McX0TFBVED+DSODkBbZ1RS1RE2nK5nQdJMKh7SLrqH7qAc8mm7x7RUTR/vQCP8tNsEW61pMfEnJx0b4FE6JnYcdg0598PQvcCxh/QlpoyBfO9CdiZOVztcGyyS5cy5pDnycmBjoGHcVfSgbQBEVoP/YZ1oZKIyLnTR0oMBe36J8s1s5I/UF4Qe/6GWDB7G+qub+89Vq7aeWl/nwBxIZEUATXrlLKj4lCnE1xtzO5MS8P/H7+Y2lOjzP+aMC/j9+qotvq0H2L8nwZv1DC63pRA2jqhXn9emYxKRlusoE5gvAHEaOvOW6BxJqkjWvSmNdaenM9sQA6iyzJn7FlzuvTc8KVx5KSC9BrKW4Ze4p7XTAEQVQMEjKCcqHXuvy5hZg0ui/fcZ8VfVDILPEwLd69mbp66B5MheeKxexZMhmGHQb+5jE2FbtrDyIb7seUrHHSRr9fB9avu0NudDi8cZmC9hNvdeZP7W0cj+E7GfEU1WeUewBcTHRLQaHgMXsf0NwWr0slGNUV1SNTPewO5p8S/mGbCe1PRZWS0I/84ymqBqoZ71HUvNQyyHtaNMnBAkWfIdkZfnYfMMoY2wV1+rzEQ+pqOTiysUTxZe2e/fleFIKLcH1WrpmaUUE4yp9jxuIwN2bDNRCG/RFFM+uL77cezFcef//VB6qs4ALsuSQ8FRImSBhZoCs9aYlHwV8T4qikP4qxMkjXAZOn+obvCUj28t1MEe/H78BDEw0PJyjZYCEUP1Vy67A6YxWrMq9fnavyLndxMXArFQW/4ek+1Y3377mYBp/yrNDQqAB636+rSJ9dcItGyP/r34+63CWpVrmqznsAIOuYBQiDrM/o8Gi3TbC4g5yKqO6+co/0oC20IzpKo/FnMk5LaCbT9qfIcXKZc3wyC1C3G/auClybHnaz96W76bn9Xrlo8C2GgXiGI1QNFYG00CTunw04JQYFX7TWkGhQ/xnFNqxklvNxExMdioSiWUV+i0AAcSoYKKSm+E+TpfsYi9onBUAF7AkDOWxvjEYbrISCcQsFHha+sawgJcjuZ84mi3XXna/aOWHXvbS3ZFmpvR2eoQ3iamjT1sw9KYGslUkp4pAN5R0Thj6AbuHuKZ7+UpFhdSXrvzB4538s5t2OZnFv/iCtFW0L6Ql4GihX0RnvYrPd8HWojTDaouByGawPyHYnoQW3GzYcg7R8JsWVofoXJXVGHCizoK0CK3qiIDcow6B45E3Zse/zHzPCiY/V6ImY7GmJVkOCdikdgrhlE9BUqa4UeFeWTD9gap8d3tE3KJze7vWnLSp0ZOKnsfuP689D/G380irfzOuXKtqdQ74pbLOJ9BN8YPz790fhjECFZdux/MM7ECssifgOFhFFibNNDJ+oSlQpu6DSV/axSKuVf/5IrlF4WOV67thtJtGUUL3jUNbc1X3u9NlXTgpo5cmhrYS3vyUZPv16Tpi2hARDERAGvP7OYZGaFThJX9nQPH8Po5c+2iiwROHf3PyoySqyabtmiJ2pqOmh0ebkuf9cuQKexhNrK6kMVh4blL/QaAIN2OS79YL5ZhTLImLTdBQiQnAu9iDhrenSk43nyVO18eFqlX/qaH9Yr2J756XhrdvKmMFIy7FCRsXBO2DmcPFMMUr6JiJ04sWsf8qk4Z4jOGQ9ieEBobK6QihtUb13znBSUGsGp2AGba7cmnln1GluSC+1zw6TCQWEZDqmh8JBTLSUDOKWtX1rJQZlIg1Yn98Lc0lVz27JM6HiN/pgzC9a+kJ3YiXKymG+DS11IuiL5dDUU4OXUVnajY47qjeQrolgO/3T0KuapyJCJDqNGUR04ZspHkiBjwfcMcOnaqaM2XB+sPfdGhIUBoFFABbMBmcwb1Tu4rlVD9MyWAFQgO5IYAm/z0hJw9Iql0Vfzc+bx0nl8PPc55LTUF8Buqtoi/6sUL6VrfLXNKarC2gPoO5SI9KOlZtREQPZ+xpQlZ6AkFVikJqD8fmQoweT7sX6h8SqekpbTPB+i6ZM96bMQawThL/5Gpf5BVv50Q0zFyfLw3IyqcOdF4fTaYb//lgVKEzDn9I/z7hCywX5OUOetvfF1CdfMPq6+ZPYhf6/KYd5ZbeSRMNDyO3/Y24v/L3vnUR+DFbHbfgQ/jUFGqxGN8mx7u0FYBM97NLjXZ8xtgQsPm1yZ6rbK7MkEcEiHVhMKLsklU4dA8cffmv3WiwH1ha3aqFfilDpDMny9lKaff9UYJm1/3jU1VEMprPD8g0NB3ickazAhIWstbABn14aAfy3zwnLqhJtye7vi4cabUlilC0HYbMiap5fIm8uSp7MmDj8RNMk4jNfT7OybWkotVQryP2jDe36X0XwgR0/RcxrZDGvhJevBXsJfZ+mT5J2suiaCZ2bc1jjBX7U3th9UDXbwzbLXb7PNdBkhbs2Td42QkfXhO19+a/znHCNA2oNjVU0kVJxxtsXTN2MkTuLE2XgQ4mXhrVGzPFoxaFx0d7jidinKYB5xNBH5dYuV3qkOjn0Y9r+ET4CHk/Cp4yho2zwvMiwuOmxRoIiu2IkNcoA0FmhRHTbQZ31+/tqPV8bF2faVnaU9kTIVTML/XCtK5HN04TAa3xFml8luY1bsvxp4G/gXQlOfFfWxKFofjsmrZabKkhu94mcGXyARFJIRChsoSN5MJv8TcpAR6v7xu6BwOh1tOd41IckrFOf22CPYLN5RqZf9mrv7AAoiq5rBPNPv/7swyVsmuRk0Kz2hcoZ4Z2CDiiGlkWb7n51AyDVeGGt2Gysp8/v4vM3WLX1d4Jw/BAIDpFWgYkMzvBzqMedhCUQL3rCxOuBHWxvP3g8GjEQSEGkI9iFJgjRcXevJ3UOg042jBWugj/Est368goVK76+yZYe/p8Z5WIkOF493L3APITLgbJO+SwXu7WdSR+t680K6oaL8WEObTttZdrKiOS/B+CnSVlEFxwGItciCmeDfTDBg15WTlfli3cZOXwMjqeKUtgZwamNa7XH/UzEhT7oFARP1bzlnq2SkaMgzeA9QWa94LGSt76qt3rQ15BBtyGnIc8/uBXgKSqg8qTpK6qyKqByl0qyJMk8I2k4Qr8zqbbH+//ShlCuRZLJNkXriao15RA/PTjQ7/xRWnqfFucvI0KE2reD1WBFU0PD48wTNH8DYTMOlFVC9ny8dvzbSwwdCTvQ4htq/O427p+6RP5ju3wlCg9P42GrYkv6l0YuOhnvxTIZ9em7RB+JFsrqHnInmC8YIGuZ6L18N04EnwStX8PZOCdvUiETjfeca2S6bTH0zCQwRLNAUylpTkzcxle3EsiICWHDWqBaYdwTRYf/6bNhr5LofXlA0Rq0KkWd4alS6Itue2Fqf80XOtAwD3kXtk/bamTZp6Jtb/e3fH1wd+rIv/sy3mVHqce8xtIaeePGAeb4rTzR7YMiYpeYaczUnHmI99lb9OSlJoaR7TYUXX64gr0h2B9pKBOFPd6rurVxO5joHNFT93IK9JRoQtpuzx+l8tAP1n2qKbq2ym4DqoW5a+jWM5tybFdUirWROw48dsNgY1Uzfxtf6m6HE7s6/LgbBqcpIrb0uUM6YgYJE/yRwQ+baEC4wbBQFBtiv1EkOYv3OW4u8zZLoq/txCKosevJvBCZOBqVQDK8/0bPARfICJd4R/2qYvHU+og4aAWMEZNoMj4tawK2lZZe4NOjRy8M0GRZVE1lr8WIScOzYGR0n/PBpSvCHIMxP0GrM945lb27DI1yM1+dqL/+itXmZY+tAM9d3IUintuYmHk7EvpvP9TH8DyK73nl1aicBCNngoQ2MsGAkb1FxpIDDTc7l758kPUtJq0cpY2f6+h29+andCY2yv+BY0UN7nsSd4ZHxeITBZ4EoHmyfxK55I13WKJpUn6bZvkk3jOvSXDY7FNUhkKZXUR7gcduJ9U1r4s5m0VEikmthH8J7DmP1/9AmBbd++vt+FcCYmIC1QyvbEzRHBp4sX9/cQxCzLJMxd6g0KfQpglBQ8G/3jksqUE1f20nF8p8yw2xTPcPNAfi9CyClcJimcbveBJJ7Acnnlk7O8+Wfo2ut5QQy3bt74Nq7OhM59tfubBCkv384Zu7exJGB3qbLpRYMY5/o2/uZdh6X+77z/MnzNPJ85ulyqUEOOdGf58j2u2n1p+pOdEWcU9facOKtTJBWMjDJWcXToieBWAN8Qc5HqyxFlLi/8cRkd8A4K8jQAsA/+v+7Bqd8qML9Qcq5r9HYwvD/1/OBF0R9P2b53+XTloJRUCnXgBCAmItBwzRgdpLQLy2A7aP881MQOAqwHULCBEOqMRNcswDvh5F90tgthWwAx3EWw6C9BBE6h3wtxKILQMKfWpzsgIgtQfw0rvRz5YBmaOAgv+g8grIvdhrSgYxzgah6s2itAKE2Q9ILAWQhXK5AyDIdBCuTcCbi0Wjfb30E4iv8ABgzP9zVgJtoE6QCwIU8TLygtsSBJOTEgyZPRJCANMkFEKKJDSJCBGTtAQGaaf4P73DVFJGPXU0E4SdUsppqVGcNJGZq2nCRaWqJZcwgglN5TgMaqWDBioCXssiMFyghLHwkaOQ9p4C1723plocGEgNNFu5J9pK1Ai5gQ4FVpKoYiCUEFAq5HBCOcofsEg5kNsWS8iKkzaCE1SVOOvPbU+uHJYKVoqtCZaQBAJAesEP0AAAAAA=); }</style></defs><rect x="0" y="0" width="525.5" height="568.5" fill="#ffffff"></rect><g stroke-linecap="round" transform="translate(10 10) rotate(0 251.5 177.5)"><path d="M32 0 C141.92 0.38, 253.1 1.58, 471 0 M32 0 C200.8 -0.74, 369.57 -1.12, 471 0 M471 0 C493.2 0.03, 504.6 11.49, 503 32 M471 0 C494.27 -0.73, 504.08 9.79, 503 32 M503 32 C502.3 90.93, 500.11 149.71, 503 323 M503 32 C502.39 109.84, 502.17 186.5, 503 323 M503 323 C502.37 345.68, 492.31 354.16, 471 355 M503 323 C504.21 343.5, 491.16 356.37, 471 355 M471 355 C348.31 354.34, 227.59 353.94, 32 355 M471 355 C339.01 356.19, 207.18 355.79, 32 355 M32 355 C12.32 355.08, -0.76 343.32, 0 323 M32 355 C9.92 354.98, 1.25 343.52, 0 323 M0 323 C1.65 214.04, 1.8 103.14, 0 32 M0 323 C-1.54 222.67, -2.55 123.27, 0 32 M0 32 C0 11.22, 10.89 -1.2, 32 0 M0 32 C0.02 11.9, 12.28 -0.97, 32 0" stroke="#1e1e1e" stroke-width="2" fill="none"></path></g><g transform="translate(35 17.5) rotate(0 65.30833435058594 37.5)"><text x="0" y="17.619999999999997" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">Image 1</text><text x="0" y="42.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Visible: true</text><text x="0" y="67.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Opacity: 1.0</text></g><g stroke-linecap="round" transform="translate(85 109) rotate(0 206 55.5)"><path d="M27.75 0 C110.61 0.65, 193.91 -0.65, 384.25 0 M27.75 0 C138.46 -0.03, 248.26 0.59, 384.25 0 M384.25 0 C403.57 1.39, 411.99 10.22, 412 27.75 M384.25 0 C402.99 1.81, 411.72 8.97, 412 27.75 M412 27.75 C410.97 46.87, 411.43 62.86, 412 83.25 M412 27.75 C411.62 46.15, 412.43 64.14, 412 83.25 M412 83.25 C410.05 102.08, 403.72 110.33, 384.25 111 M412 83.25 C410.04 102.7, 403.97 111.12, 384.25 111 M384.25 111 C272.77 110, 158.38 108.96, 27.75 111 M384.25 111 C287.34 112.9, 190.3 113.78, 27.75 111 M27.75 111 C7.26 112.37, 0.78 100.35, 0 83.25 M27.75 111 C9.61 110.93, 0.64 99.73, 0 83.25 M0 83.25 C-0.45 66.32, 0.72 48.96, 0 27.75 M0 83.25 C-0.77 67.35, -0.43 51.49, 0 27.75 M0 27.75 C-0.61 9.57, 10.61 1.05, 27.75 0 M0 27.75 C-0.58 9.96, 10.26 -2.03, 27.75 0" stroke="#1e1e1e" stroke-width="2" fill="none"></path></g><g transform="translate(90 114) rotate(0 90.6500015258789 50)"><text x="0" y="17.619999999999997" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">Channel 1</text><text x="0" y="42.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Visible: true</text><text x="0" y="67.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Color: Red</text><text x="0" y="92.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Window: [0, 255]</text></g><g stroke-linecap="round" transform="translate(88 234.5) rotate(0 206 55.5)"><path d="M27.75 0 C132.94 -1.14, 240.03 -0.44, 384.25 0 M27.75 0 C120.15 0.58, 212.86 -0.47, 384.25 0 M384.25 0 C404.69 -1.05, 411.47 9.63, 412 27.75 M384.25 0 C404.67 0.32, 413.09 6.96, 412 27.75 M412 27.75 C413.53 48.02, 411.08 68.04, 412 83.25 M412 27.75 C411.97 42.25, 411.83 54.54, 412 83.25 M412 83.25 C412.72 101.24, 401.45 112.23, 384.25 111 M412 83.25 C409.9 100.97, 401.02 112.41, 384.25 111 M384.25 111 C260.8 109.61, 136.85 109.58, 27.75 111 M384.25 111 C302.97 112.83, 220.5 112.1, 27.75 111 M27.75 111 C10.35 109.32, -0.3 101.55, 0 83.25 M27.75 111 C10.23 112.88, -0.4 101.27, 0 83.25 M0 83.25 C0.59 60.74, -1.12 41.51, 0 27.75 M0 83.25 C-0.53 62.46, 1.02 40.85, 0 27.75 M0 27.75 C-0.57 9.99, 9.23 1.02, 27.75 0 M0 27.75 C-0.78 9.06, 8.17 -0.64, 27.75 0" stroke="#1e1e1e" stroke-width="2" fill="none"></path></g><g transform="translate(93 239.5) rotate(0 90.9749984741211 50)"><text x="0" y="17.619999999999997" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">Channel 2</text><text x="0" y="42.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Visible: false</text><text x="0" y="67.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Color: Yellow</text><text x="0" y="92.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Window: [24, 99]</text></g><g stroke-linecap="round" transform="translate(12.5 389.5) rotate(0 251.5 84.5)"><path d="M32 0 C120.89 -2.14, 210.27 -0.95, 471 0 M32 0 C164.79 -0.68, 298.27 -0.43, 471 0 M471 0 C492.06 -0.28, 504.09 9.75, 503 32 M471 0 C492.5 1.44, 501.78 11.52, 503 32 M503 32 C503.66 55.06, 502.72 80.39, 503 137 M503 32 C503.9 58.15, 503.62 84.36, 503 137 M503 137 C504.82 159.02, 490.49 168.36, 471 169 M503 137 C503.85 160.56, 491.94 168.92, 471 169 M471 169 C375.03 168.59, 279.89 168.48, 32 169 M471 169 C376.25 168.84, 281.95 169.01, 32 169 M32 169 C9.96 167.57, 1.53 157.89, 0 137 M32 169 C9.68 170.49, -1.01 158.88, 0 137 M0 137 C-0.41 114.34, -1.17 92.58, 0 32 M0 137 C-0.2 110.82, 0.73 85.09, 0 32 M0 32 C-0.73 10.02, 10.26 -0.83, 32 0 M0 32 C-0.02 11.12, 9.63 -1.07, 32 0" stroke="#1e1e1e" stroke-width="2" fill="none"></path></g><g transform="translate(36.69166564941406 401.5) rotate(0 66.82499694824219 37.5)"><text x="0" y="17.619999999999997" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">Image 2</text><text x="0" y="42.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Visible: true</text><text x="0" y="67.62" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">- Opacity: 0.5</text></g><g stroke-linecap="round" transform="translate(87 493.5) rotate(0 206 26.5)"><path d="M13.25 0 C115.54 -1.23, 216.75 0.03, 398.75 0 M13.25 0 C94.2 1.63, 174.95 0.94, 398.75 0 M398.75 0 C407.76 0.59, 412.39 2.68, 412 13.25 M398.75 0 C407.4 -0.78, 413.69 3.68, 412 13.25 M412 13.25 C410.66 19.17, 410.1 26.11, 412 39.75 M412 13.25 C412.65 18.4, 412.37 24, 412 39.75 M412 39.75 C412.94 48.02, 409.42 52.38, 398.75 53 M412 39.75 C409.96 49.45, 405.62 55.26, 398.75 53 M398.75 53 C261.58 52.87, 124.04 53.03, 13.25 53 M398.75 53 C253.79 55.72, 108.53 54.69, 13.25 53 M13.25 53 C4.37 51.77, 0.59 48.65, 0 39.75 M13.25 53 C5.16 51.29, 0.41 50.33, 0 39.75 M0 39.75 C-1.98 31.74, -0.99 21.12, 0 13.25 M0 39.75 C-1.11 31.32, -0.45 21.45, 0 13.25 M0 13.25 C0.44 6.27, 5.28 0.39, 13.25 0 M0 13.25 C1.42 3.9, 2.86 -0.11, 13.25 0" stroke="#1e1e1e" stroke-width="2" fill="none"></path></g><g transform="translate(92 498.5) rotate(0 8.225000381469727 12.5)"><text x="0" y="17.619999999999997" font-family="Excalifont, Xiaolai, sans-serif, Segoe UI Emoji" font-size="20px" fill="#1e1e1e" text-anchor="start" style="white-space: pre;" direction="ltr" dominant-baseline="alphabetic">...</text></g></svg>
"""))

The `link_views_by_dict()` method with the `CoordinationLevel` (CL) function allows you to specify these types of hierarchical properties in the initial configuration.

The example below uses a [CyCIF](https://www.cycif.org/) multiplexed immunofluorescence dataset (outputs of the [MCMICRO](https://mcmicro.org/) pipeline). We configure the initial channel colors.

Similar to `VitessceConfig.link_views`, the first parameter of `VitessceConfig.link_views_by_dict` is an array of views for which you want to specify the coordination values. The second parameter of `link_views_by_dict` is a Python dictionary mapping coordination types to their initial values.

In [ ]:
mcmicro_zip = join(data_dir, "mcmicro_io.spatialdata.zarr.zip")
mcmicro_path = join(data_dir, "mcmicro_io.spatialdata.zarr")

if not isdir(mcmicro_path):
    if not isfile(mcmicro_zip):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/mcmicro_io.spatialdata.zarr.zip",
            mcmicro_zip
        )
    with zipfile.ZipFile(mcmicro_zip, "r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), mcmicro_path)

sdata_mc = read_zarr(mcmicro_path)
sdata_mc

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    CoordinationLevel as CL,
    SpatialDataWrapper,
    get_initial_coordination_scope_prefix,
)

vc_mc = VitessceConfig(schema_version="1.0.18", name="Multiplexed Image (CyCIF)")

mc_wrapper = SpatialDataWrapper(
    sdata_store=mcmicro_path,
    image_path="images/exemplar-001_image",
    coordinate_system="global",
    coordination_values={
        "obsType": "cell"
    },
)
dataset_mc = vc_mc.add_dataset(name="MCMicro CyCIF").add_object(mc_wrapper)

spatial_mc = vc_mc.add_view("spatialBeta", dataset=dataset_mc)

# Set initial channel colors and visibility
vc_mc.link_views_by_dict(
    [spatial_mc],
    {
        "imageLayer": CL([{
            "photometricInterpretation": "BlackIsZero",
            "imageChannel": CL([
                {"spatialTargetC": 0, "spatialChannelColor": [255, 0, 0],   "spatialChannelOpacity": 1.0},
                {"spatialTargetC": 1, "spatialChannelColor": [0, 255, 0],   "spatialChannelOpacity": 1.0},
                {"spatialTargetC": 2, "spatialChannelColor": [0, 0, 255],   "spatialChannelOpacity": 1.0},
            ]),
        }]),
    },
    scope_prefix=get_initial_coordination_scope_prefix("A", "image"),
)

vc_mc.layout(spatial_mc)
vc_mc.widget()

#### Exercise

Now, try modifying the above configuration to specify better channel intensity window ranges. (Hint: try `spatialChannelWindow`)

## More spatial layers

We often want to render multiple data layers rendered simultaneously: the tissue image, cell segmentation outlines, spot circles, and molecule point clouds. Vitessce supports all of these in the same spatial view via the `layerControllerBeta`.

The table below shows the `SpatialDataWrapper` parameter for each layer type:

| Layer Type | What it shows | `SpatialDataWrapper` param |
|---|---|---|
| Image | Tissue photograph or multiplex image | `image_path` |
| Spots | Visium capture spots (circles) | `obs_spots_path` |
| Segmentation polygons | Cell outlines (GeoDataFrame shapes) | `obs_shapes_path` |
| Segmentation bitmask | Integer label image where each value = one cell | `obs_segmentations_path` |
| Molecule points | Sub-cellular transcript coordinates | `obs_points_path` |

The [SpatialData Blobs](https://spatialdata.scverse.org/en/stable/api/datasets.html#spatialdata.datasets.blobs) toy dataset bundles several of these together and is useful for learning. Let's build a multi-layer view with it.

In [ ]:
import spatialdata
import pandas as pd
from anndata import AnnData

# Create the blobs dataset in memory
sdata_blobs = spatialdata.datasets.blobs()
sdata_blobs

Create a `Table` with a `var` dataframe corresponding to the `genes` column of the Points table

In [ ]:
ddf = sdata_blobs.points['blobs_points']
ddf

In [ ]:
# The current `table` element contains a `var` dataframe corresponding
# to the 3 image channels, rather than the 2 gene IDs.
print(sdata_blobs.tables['table'].var.index.tolist())

In [ ]:
# We need to create another table which has a var.index column containing the gene IDs for the points.
unique_gene_ids = ddf["genes"].unique().compute().tolist()
points_var_df = pd.DataFrame(index=unique_gene_ids, data=[], columns=[])
points_table = AnnData(var=points_var_df, obs=None, X=None)
sdata_blobs.tables['table_points'] = points_table

In [ ]:
unique_gene_ids

In [ ]:
sdata_blobs.write('data/blobs.sdata.zarr', overwrite=True)

In [ ]:
sdata_blobs

Note the addition of the `table_points` element in the updated SpatialData object.

In [ ]:
from vitessce import SpatialDataWrapper, CoordinationLevel as CL, get_initial_coordination_scope_prefix

vc_blobs = VitessceConfig(schema_version="1.0.18", name="Blobs: Image + Labels + Points")

# Wrapper 1: image + segmentation bitmask (labels)
blobs_wrapper = SpatialDataWrapper(
    sdata_store="data/blobs.sdata.zarr",
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    image_path="images/blobs_multiscale_image",
    obs_segmentations_path="labels/blobs_labels",
    obs_embedding_paths=["tables/table/obsm/X_umap"],
    obs_feature_matrix_path="tables/table/X",
    coordinate_system="global",
    coordination_values={
        "obsType": "blob",
        "featureType": "channel",
        "fileUid": "my_unique_id"
    }
)
# Wrapper 2: molecule points
points_wrapper = SpatialDataWrapper(
    sdata_store="data/blobs.sdata.zarr",
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    obs_points_path="points/blobs_points",
    obs_feature_matrix_path="tables/table_points/X", # TODO
    coordinate_system="global",
    coordination_values={
        "obsType": "point",
        "featureType": "gene",
        "fileUid": "other_unique_id"
    }
)
dataset_blobs = vc_blobs.add_dataset(name="Blobs").add_object(blobs_wrapper).add_object(points_wrapper)

spatial_blobs = vc_blobs.add_view("spatialBeta", dataset=dataset_blobs)

vc_blobs.link_views_by_dict([spatial_blobs], {
    'imageLayer': CL([{
        "fileUid": "my_unique_id",
        'photometricInterpretation': 'BlackIsZero',
        'spatialLayerOpacity': 0.9,
        'imageChannel': CL([
            {
                "spatialTargetC": 0,
                "spatialChannelColor": [255, 0, 0],
                "spatialChannelOpacity": 1.0,
                "spatialChannelWindow": [0.0, 1.0],
            },
            {
                "spatialTargetC": 1,
                "spatialChannelColor": [0, 255, 0],
                "spatialChannelOpacity": 1.0,
                "spatialChannelWindow": [0.0, 1.0],
            },
            {
                "spatialTargetC": 2,
                "spatialChannelColor": [0, 0, 255],
                "spatialChannelOpacity": 1.0,
                "spatialChannelWindow": [0.0, 1.0],
            }
        ])
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "image"))

vc_blobs.link_views_by_dict([spatial_blobs], {
    'segmentationLayer': CL([{
        "fileUid": "my_unique_id",
        'segmentationChannel': CL([{
            'spatialChannelVisible': True,
            'obsType': 'blob',
            "featureType": "channel",
        }]),
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsSegmentations"))

vc_blobs.link_views_by_dict([spatial_blobs], {
    'pointLayer': CL([{
        "fileUid": "other_unique_id",
        "obsType": "point",
        "featureType": "gene",
        "obsHighlight": None,
        "obsColorEncoding": "randomByFeature",
        "spatialLayerOpacity": 1.0,
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsPoints"))

vc_blobs.layout(spatial_blobs)
vc_blobs.widget()

## Simplifying things with SpatialData-Plot and EasyVitessce

The low-level `vitessce` API is powerful but verbose. [EasyVitessce](https://vitessce.github.io/easy_vitessce/) intercepts [SpatialData-Plot](https://spatialdata.scverse.org/projects/plot/en/latest/) plotting calls and automatically generates an interactive Vitessce widget instead of returning a static figure.

### How it works

1. Install and **`import easy_vitessce as ev`**. Running this import line itself __automatically__ enables the interactive output (with the option to disable later if needed)
2. Use the familiar `sdata.pl.render_*().pl.show()` chain. (If unfamiliar with SpatialData-Plot, see its [documentation](https://spatialdata.scverse.org/projects/plot/en/latest/))
3. The widget is live: pan, zoom, select cells, change channels

This means existing SpatialData-Plot workflows become interactive with just one extra import.

In [ ]:
!pip install "spatialdata-plot==0.4.0"

In [ ]:
!pip install "easy_vitessce==0.0.12"

In [ ]:
import easy_vitessce as ev
import spatialdata_plot

# This line is required when the notebook kernel is running on a different machine (e.g., Google Colab, Docker container, or HPC cluster)
ev.config.set({ 'data.wrapper_param_suffix': '_store' })

# Re-use the Visium SpatialData object loaded earlier
(
    sdata
        .pl.render_images("ST8059050_hires_image")
        .pl.render_shapes("ST8059050", color="Fth1", table_name="table")
        .pl.show("ST8059050")
)

Compared to the `SpatialDataWrapper` example earlier, the EasyVitessce version is much more concise. The parameters map directly to SpatialData-Plot conventions:

- `.pl.render_images("element_name")`: add an image layer
- `.pl.render_shapes("element_name", color="GeneName", table_name="table")`: add a spot or segmentation layer colored by gene expression
- `.pl.show("coordinate_system")` — render and display the widget

If you are curious, you can check out the EasyVitessce [source code](https://github.com/vitessce/easy_vitessce/blob/main/src/easy_vitessce/spatialdata_plot.py) which internally translates the SpatialData-Plot syntax to the Vitessce configuration code we learned above.

### Static vs interactive

You can switch between static matplotlib output and the interactive Vitessce widget at any time:

In [ ]:
# Disable interactive plots: renders with static matplotlib
ev.disable_plots(["spatialdata-plot"])

(
    sdata
        .pl.render_images("ST8059050_hires_image")
        .pl.render_shapes("ST8059050", color="Fth1", table_name="table")
        .pl.show("ST8059050")
)

In [ ]:
# Re-enable interactive plots for the exercise below
ev.enable_plots(["spatialdata-plot"])

### Exercise 2

👉 **Modify the EasyVitessce cell above** to:

- **Change** the `color` parameter from `"Fth1"` to a different gene (Note: this is a mouse dataset)
- **Add** a segmentation label layer using `.pl.render_labels("element_name")` before `.pl.show()`


In [ ]:
sdata.tables["table"].var

In [ ]:
# Your answer here
(
    sdata
        .pl.render_images("ST8059050_hires_image")
        .pl.render_shapes("ST8059050", color="red", table_name="table")
        .pl.show("ST8059050")
)

Finally, check out the EasyVitessce [documentation](https://vitessce.github.io/easy_vitessce/easy_vitessce.html#easy_vitessce.spatialdata_plot.VitesscePlotAccessor.render_shapes) to learn more about the available functions and parameters.